# Cadenas de Markov: ratón y serpientes y escaleras

## Problema 1. El ratón en el laberinto

El ratón se mueve en un laberinto de 3 por 3. La casilla 7 tiene comida y la casilla 8 una descarga eléctrica. En cada paso se elige al azar una de las casillas vecinas. Se quiere saber la probabilidad de que el ratón, empezando en la casilla 0, llegue a la comida.

### Parte analítica

Sea $p_i$ la probabilidad de llegar a la comida empezando en la casilla $i$. Se sabe que $p_7 = 1$ y $p_8 = 0$. Para las demás casillas se cumple que $p_i$ es el promedio de los $p_j$ de sus vecinos. Esto da un sistema de 7 ecuaciones con 7 incógnitas que se resuelve con numpy.

In [1]:
import numpy as np
import random

# Vecinos de cada casilla del laberinto
vecinos = {
    0: [1, 2],
    1: [0, 3, 7],
    2: [0, 3, 8],
    3: [1, 2, 4, 5],
    4: [3, 6, 7],
    5: [3, 6, 8],
    6: [4, 5]
}

In [2]:
# Se arma el sistema A p = b para las casillas 0 a 6

A = np.eye(7)
b = np.zeros(7)

for i in range(7):
    grado = len(vecinos[i])
    for j in vecinos[i]:
        if j == 7:
            b[i] = b[i] + 1/grado
        elif j == 8:
            pass
        else:
            A[i, j] = A[i, j] - 1/grado

p = np.linalg.solve(A, b)

for i in range(7):
    print("p_" + str(i) + " =", round(p[i], 4))

print()
print("Probabilidad de llegar a la comida desde la casilla 0:", round(p[0], 4))

p_0 = 0.5
p_1 = 0.6667
p_2 = 0.3333
p_3 = 0.5
p_4 = 0.6667
p_5 = 0.3333
p_6 = 0.5

Probabilidad de llegar a la comida desde la casilla 0: 0.5


### Parte por simulación

Se simulan muchas trayectorias del ratón empezando en la casilla 0. Cada trayectoria se detiene al llegar a la 7 o a la 8. Al final se cuenta cuántas veces se llegó a la comida.

In [3]:
random.seed(42)

n = 100000
exitos = 0

for k in range(n):
    pos = 0
    while pos != 7 and pos != 8:
        pos = random.choice(vecinos[pos])
    if pos == 7:
        exitos = exitos + 1

prob_simulada = exitos / n

print("Probabilidad simulada:", round(prob_simulada, 4))
print("Probabilidad analitica:", round(p[0], 4))

Probabilidad simulada: 0.5012
Probabilidad analitica: 0.5


El resultado analítico es $p_0 = 1/2$ y la simulación da un valor muy cercano. Esto tiene sentido por la simetría del laberinto: la casilla 7 y la 8 están en esquinas opuestas, así que desde la casilla 0 las dos son igual de probables.

## Problema 2. Serpientes y escaleras

El tablero tiene 20 casillas. Se empieza fuera del tablero (casilla 0) y en cada turno se lanza un dado de seis caras. Si se cae en una escalera se sube, y si se cae en una serpiente se baja. El juego acaba al llegar a la casilla 20.

Del tablero se leen las siguientes escaleras y serpientes: escalera de 3 a 11, escalera de 15 a 19, serpiente de 17 a 9 y serpiente de 13 a 5. Se quiere el número promedio de tiradas para terminar el juego.

### Parte analítica

Sea $E_i$ el número esperado de tiradas para terminar empezando en la casilla $i$. Se cumple $E_{20} = 0$ y para las demás casillas

$$E_i = 1 + \frac{1}{6}\sum_{d=1}^{6} E_{f(i+d)},$$

donde $f$ aplica las escaleras y serpientes. Esto se reescribe como un sistema lineal y se resuelve con numpy.

In [4]:
# Escaleras y serpientes: de donde sale -> a donde llega
saltos = {3: 11, 15: 19, 17: 9, 13: 5}

def mover(pos):
    if pos in saltos:
        return saltos[pos]
    else:
        return pos

In [5]:
# Se construye la matriz de transicion P de 21 por 21

P = np.zeros((21, 21))

for s in range(20):
    for d in range(1, 7):
        nueva = s + d
        if nueva >= 20:
            nueva = 20
        else:
            nueva = mover(nueva)
        P[s, nueva] = P[s, nueva] + 1/6

P[20, 20] = 1

# Se verifica que las filas sumen 1
print("Suma de filas:", np.round(P.sum(axis=1), 2))

Suma de filas: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
# Se resuelve (I - Q) E = 1 para las casillas 0 a 19
Q = P[:20, :20]
I = np.eye(20)
unos = np.ones(20)

E = np.linalg.solve(I - Q, unos)

print("Numero esperado de tiradas desde el estado 0:", round(E[0], 4))

Numero esperado de tiradas desde el estado 0: 7.0521


### Parte por simulación

Se simulan muchas partidas completas y se promedia el número de tiradas.

In [7]:
random.seed(123)

n = 100000
total = 0

for k in range(n):
    pos = 0
    tiradas = 0
    while pos < 20:
        d = random.randint(1, 6)
        nueva = pos + d
        if nueva >= 20:
            nueva = 20
        else:
            nueva = mover(nueva)
        pos = nueva
        tiradas = tiradas + 1
    total = total + tiradas

promedio = total / n

print("Promedio simulado:", round(promedio, 4))
print("Valor analitico: ", round(E[0], 4))

Promedio simulado: 7.0455
Valor analitico:  7.0521


## Conclusión

Para el ratón se obtuvo que la probabilidad de llegar a la comida desde la casilla 0 es $1/2$. Para el juego de serpientes y escaleras el número promedio de tiradas para terminar es aproximadamente $7.05$. En los dos problemas la simulación coincide con el resultado analítico.